In [ ]:
"""
Amazon Product Reviews Sentiment Analysis - Testing Script
Load trained model and predict sentiment for new reviews
"""

import pickle
import re
import warnings
warnings.filterwarnings('ignore')


class SentimentPredictor:
    """
    Load and use trained sentiment analysis model
    """
    
    def __init__(self, model_path='sentiment_model_final.pkl'):
        """
        Load the trained model
        """
        print("="*70)
        print("SENTIMENT ANALYSIS - MODEL TESTING")
        print("="*70)
        print(f"\nLoading model from {model_path}...")
        
        try:
            with open(model_path, 'rb') as f:
                model_data = pickle.load(f)
            
            self.model = model_data['model']
            self.vectorizer = model_data['vectorizer']
            self.model_name = model_data['model_name']
            self.class_names = model_data['class_names']
            self.stop_words = model_data['stop_words']
            
            print(f"✓ Model loaded successfully!")
            print(f"  - Algorithm: {self.model_name}")
            print(f"  - Classes: {len(self.class_names)}")
            print("\nClass Labels:")
            for class_id, class_name in self.class_names.items():
                print(f"  {class_id}: {class_name}")
            
        except FileNotFoundError:
            print(f"❌ Error: Model file '{model_path}' not found!")
            print("Please run the training script first: sentiment_analysis_training.py")
            raise
    
    def preprocess_text(self, text):
        """
        Preprocess text same as training
        """
        # Convert to lowercase
        text = text.lower()
        
        # Remove URLs
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        
        # Remove HTML tags
        text = re.sub(r'<.*?>', '', text)
        
        # Remove special characters and numbers
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Tokenize and remove stopwords
        words = text.split()
        words = [word for word in words if word not in self.stop_words and len(word) > 2]
        
        # Simple suffix stripping (basic stemming)
        stemmed_words = []
        for word in words:
            if word.endswith('ing'):
                word = word[:-3]
            elif word.endswith('ed'):
                word = word[:-2]
            elif word.endswith('ly'):
                word = word[:-2]
            stemmed_words.append(word)
        
        return ' '.join(stemmed_words)
    
    def predict(self, review):
        """
        Predict sentiment for a single review
        """
        # Preprocess
        processed_review = self.preprocess_text(review)
        
        # Vectorize
        review_vector = self.vectorizer.transform([processed_review])
        
        # Predict
        prediction = self.model.predict(review_vector)[0]
        prediction_proba = self.model.predict_proba(review_vector)[0]
        
        # Get sentiment label
        sentiment = self.class_names[prediction]
        confidence = prediction_proba[prediction] * 100
        
        return {
            'review': review,
            'prediction': prediction,
            'sentiment': sentiment,
            'confidence': confidence,
            'probabilities': {
                self.class_names[i]: prob * 100 
                for i, prob in enumerate(prediction_proba)
            }
        }
    
    def predict_batch(self, reviews):
        """
        Predict sentiment for multiple reviews
        """
        results = []
        for review in reviews:
            result = self.predict(review)
            results.append(result)
        return results
    
    def display_prediction(self, result):
        """
        Display prediction results in a formatted way
        """
        print("\n" + "="*70)
        print("PREDICTION RESULT")
        print("="*70)
        print(f"\n📝 Review:")
        print(f"   {result['review']}")
        print(f"\n🎯 Prediction: {result['sentiment']} (Class {result['prediction']})")
        print(f"   Confidence: {result['confidence']:.2f}%")
        print(f"\n📊 Probability Distribution:")
        for sentiment, prob in result['probabilities'].items():
            bar_length = int(prob / 2)  # Scale to 50 chars max
            bar = '█' * bar_length
            print(f"   {sentiment:15s} [{prob:5.2f}%] {bar}")
        print("="*70)
    
    def interactive_test(self):
        """
        Interactive testing mode
        """
        print("\n" + "="*70)
        print("INTERACTIVE TESTING MODE")
        print("="*70)
        print("\nEnter product reviews to analyze sentiment.")
        print("Type 'quit' to exit.\n")
        
        while True:
            review = input("Enter product review: ").strip()
            
            if review.lower() == 'quit':
                print("\nExiting interactive mode. Goodbye!")
                break
            
            if not review:
                print("Please enter a valid review.\n")
                continue
            
            result = self.predict(review)
            self.display_prediction(result)
            print()


def test_sample_reviews():
    """
    Test with predefined sample reviews
    """
    print("\n" + "="*70)
    print("TESTING WITH SAMPLE REVIEWS")
    print("="*70)
    
    # Load model
    predictor = SentimentPredictor('sentiment_model_final.pkl')
    
    # Sample test reviews
    test_reviews = [
        "This product is absolutely amazing! Best purchase I've ever made. Highly recommend!",
        "Good quality product. Works as expected. Happy with the purchase.",
        "It's okay. Nothing special but does the job.",
        "Not satisfied. Quality is poor and doesn't work properly.",
        "Terrible! Completely broke after 2 days. Total waste of money!",
        "The product arrived on time and packaging was good. Performance is decent.",
        "Outstanding! Exceeded all my expectations. Worth every penny!",
        "Disappointed with this. Expected much better for the price.",
        "Fantastic product! Amazing quality and great customer service!",
        "Below average. Would not recommend to anyone."
    ]
    
    print(f"\nTesting {len(test_reviews)} sample reviews...\n")
    
    # Predict and display
    for i, review in enumerate(test_reviews, 1):
        result = predictor.predict(review)
        
        print(f"\n{'─'*70}")
        print(f"Test Review #{i}")
        print(f"{'─'*70}")
        print(f"Review: {review}")
        print(f"Prediction: {result['sentiment']} (Confidence: {result['confidence']:.1f}%)")
    
    print("\n" + "="*70)
    print("SAMPLE TESTING COMPLETED")
    print("="*70)


def demonstrate_detailed_analysis():
    """
    Detailed analysis of specific reviews
    """
    print("\n" + "="*70)
    print("DETAILED SENTIMENT ANALYSIS DEMONSTRATION")
    print("="*70)
    
    predictor = SentimentPredictor('sentiment_model_final.pkl')
    
    # Examples representing each class
    detailed_reviews = [
        "This is the worst product I've ever bought! Absolute garbage! Broke immediately!",
        "Poor quality. Not worth the money. Several defects found.",
        "Average product. Neither impressed nor disappointed. Just okay.",
        "Really good! Works perfectly and great value for money. Recommend it!",
        "AMAZING! PERFECT! Best product ever! Can't be happier! 10/10!"
    ]
    
    for review in detailed_reviews:
        result = predictor.predict(review)
        predictor.display_prediction(result)


if __name__ == "__main__":
    try:
        # Run sample tests
        test_sample_reviews()
        
        # Run detailed analysis
        demonstrate_detailed_analysis()
        
        # Interactive mode option
        print("\n" + "="*70)
        print("Would you like to enter interactive testing mode?")
        print("="*70)
        choice = input("\nEnter 'yes' to start interactive mode (or anything else to exit): ")
        
        if choice.lower() in ['yes', 'y']:
            predictor = SentimentPredictor('sentiment_model_final.pkl')
            predictor.interactive_test()
        else:
            print("\nThank you for using the Sentiment Analysis System!")
            print("="*70)
    
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("\nMake sure to run 'sentiment_analysis_training.py' first to train the model.")



TESTING WITH SAMPLE REVIEWS
SENTIMENT ANALYSIS - MODEL TESTING

Loading model from sentiment_model_final.pkl...
✓ Model loaded successfully!
  - Algorithm: Logistic Regression
  - Classes: 5

Class Labels:
  0: Very Negative
  1: Negative
  2: Neutral
  3: Positive
  4: Very Positive

Testing 10 sample reviews...


──────────────────────────────────────────────────────────────────────
Test Review #1
──────────────────────────────────────────────────────────────────────
Review: This product is absolutely amazing! Best purchase I've ever made. Highly recommend!
Prediction: Very Positive (Confidence: 76.2%)

──────────────────────────────────────────────────────────────────────
Test Review #2
──────────────────────────────────────────────────────────────────────
Review: Good quality product. Works as expected. Happy with the purchase.
Prediction: Positive (Confidence: 77.4%)

──────────────────────────────────────────────────────────────────────
Test Review #3
───────────────────────────


Enter 'yes' to start interactive mode (or anything else to exit):  yes


SENTIMENT ANALYSIS - MODEL TESTING

Loading model from sentiment_model_final.pkl...
✓ Model loaded successfully!
  - Algorithm: Logistic Regression
  - Classes: 5

Class Labels:
  0: Very Negative
  1: Negative
  2: Neutral
  3: Positive
  4: Very Positive

INTERACTIVE TESTING MODE

Enter product reviews to analyze sentiment.
Type 'quit' to exit.



Enter product review:  Damage product 



PREDICTION RESULT

📝 Review:
   Damage product

🎯 Prediction: Positive (Class 3)
   Confidence: 24.57%

📊 Probability Distribution:
   Very Negative   [23.76%] ███████████
   Negative        [11.25%] █████
   Neutral         [19.19%] █████████
   Positive        [24.57%] ████████████
   Very Positive   [21.24%] ██████████



Enter product review:  AMAZING! PERFECT! Best product ever! Can't be happier! 



PREDICTION RESULT

📝 Review:
   AMAZING! PERFECT! Best product ever! Can't be happier!

🎯 Prediction: Very Positive (Class 4)
   Confidence: 70.67%

📊 Probability Distribution:
   Very Negative   [ 9.23%] ████
   Negative        [ 6.93%] ███
   Neutral         [ 6.14%] ███
   Positive        [ 7.03%] ███
   Very Positive   [70.67%] ███████████████████████████████████



Enter product review:   Poor quality. Not worth the money. Several defects found.



PREDICTION RESULT

📝 Review:
   Poor quality. Not worth the money. Several defects found.

🎯 Prediction: Negative (Class 1)
   Confidence: 79.25%

📊 Probability Distribution:
   Very Negative   [ 5.34%] ██
   Negative        [79.25%] ███████████████████████████████████████
   Neutral         [ 4.27%] ██
   Positive        [ 4.38%] ██
   Very Positive   [ 6.76%] ███

